In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
patient_table = dbutils.widgets.get("patient_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")
icd_table = dbutils.widgets.get("icd_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW diagnosis_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(Sequence AS INT) AS Sequence,
  CAST(DiagCode AS STRING) AS DiagCode,
  CAST(DiagDesc AS STRING) AS DiagDesc,
  CAST(Version AS INT) AS Version,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  diagnosis_cte AS (
    SELECT
      CAST('{fetch_date}' AS DATE) AS ReportingDate,
      b.ExternalId AS FacilityCode,
      bl.ClaimNumber AS AcctNbr,
      '1' AS Sequence,
      REPLACE(REPLACE(icd.Icd9Cm, '\r', ''), '\n', '') AS DiagCode,
      ' ' AS DiagDesc,
      '10' AS Version,
      '19' AS SourceSystemKey
    FROM {source_table} bl
    JOIN {patientpayer_table} pp ON pp.id = bl.PatientPayerId
    JOIN {patient_table} p ON p.id = pp.PatientId
    JOIN {branch_table} b ON b.id = p.BranchId
    JOIN {icd_table} icd ON icd.PatientId = p.Id 
    AND icd.IsPrimary = true --needed if sequence is only 1
    WHERE bl.isActive='true'
  ),
  diagnosis_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM diagnosis_cte
  )
  SELECT 
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Sequence,
    DiagCode,
    DiagDesc,
    Version,
    SourceSystemKey
  FROM diagnosis_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING diagnosis_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Sequence = src.Sequence,
    tgt.DiagCode = src.DiagCode,
    tgt.DiagDesc = src.DiagDesc,
    tgt.Version = src.Version,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Sequence,
    DiagCode,
    DiagDesc,
    Version,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Sequence,
    src.DiagCode,
    src.DiagDesc,
    src.Version,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)